<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# Build a Chatbot Interface for the Recommendation System


Estimated time needed: **45** minutes


## Learning Objectives
After completing this lab, you'll be able to:

- Build interactive chatbot interfaces using Gradio
- Implement natural language intent classification for user requests
- Extract user preferences from conversational input
- Integrate multi-agent workflows into a chat interface
- Format agent outputs for display in a user-friendly format
- Add database management features to the chatbot
- Deploy and test a complete chatbot application


## Introduction


In this lab, you will build a chatbot interface for your multi-agent recommendation system using Gradio. The chatbot will:

1. Understand user requests in natural language
2. Classify whether users want restaurant or recipe recommendations
3. Extract preferences like cuisines, dietary restrictions, and price range
4. Invoke the multi-agent workflow from Lesson 2
5. Present recommendations in a conversational, engaging format
6. Allow users to add, update, and delete items in the database

By the end of this lab, you will have a fully functional chatbot that anyone can use without writing code.

**Note:** Throughout this lab, you'll answer strategically placed questions designed to reinforce your learning.


## Table of Contents

<font size = 3>    

1. [Install the required libraries](#Install-the-required-libraries)
2. [Import required libraries](#Import-required-libraries)
3. [Build a basic chatbot](#Build-a-basic-chatbot)
4. [Question 1: What two parameters does the chatbot function receive from Gradio's ChatInterface?](#Question-1:-What-two-parameters-does-the-chatbot-function-receive-from-Gradio's-ChatInterface?)
5. [Implement intent classification](#Implement-intent-classification)
6. [Extract user preferences](#Extract-user-preferences)
    1. [Task 1: Test the preference extraction function](#Task-1:-Test-the-preference-extraction-function)
7. [Integrate the multi-agent workflow](#Integrate-the-multi-agent-workflow)
8. [Format the recommendations](#Format-the-recommendations)
9. [Build a complete chatbot](#Build-a-complete-chatbot)
10. [Question 2: What are the five possible intent classifications the chatbot can identify?](#Question-2:-What-are-the-five-possible-intent-classifications-the-chatbot-can-identify?)
11. [Add database management](#Add-database-management)
12. [Deploy and test](#Deploy-and-test)
13. [Launch the chatbot](#Launch-the-chatbot)
14. [Question 3: What are the four tabs in the complete Gradio interface?](#Question-3:-What-are-the-four-tabs-in-the-complete-Gradio-interface?)
15. [Test the chatbot](#Test-the-chatbot)
     1. [Task 2: Test the chatbot with a recipe request](#Task-2:-Test-the-chatbot-with-a-recipe-request)


</font>
</div>


## **Screenshot requirement for this lab**

You will be prompted to take a screenshot and save it on your own device. You will need this screenshot either to answer graded quiz questions or to upload as your submission for the Final Project at the end of this course. You can use various free screen-grabbing tools or your operating system's shortcut keys to do this (for example, `Alt+PrintScreen` on Windows and `Command+shift+4` on Mac).
**Note**: The screenshot can be saved with either the **.jpg** or **.png** extension.


### Install the required libraries

All the required libraries are __not__ pre-installed in the Skills Network Labs environment. __You need to run the following cell__ to install them, and this might take a few minutes.
**Restart the Kernel** once the install completed.


----


In [1]:
%%capture
# This notebook needs Gradio for the UI, the OpenAI client (pointed at the local
# Ollama endpoint), and — now that the chat tab calls the real M3L2 workflow —
# the retrieval stack that M2L1 installed.
#
# The original version of this cell also pinned langchain==0.2.16,
# langchain-openai==0.1.7, langchain-core==0.2.38 and langgraph==0.1.19. None of
# them are imported here, and those pins force numpy<2, which breaks scipy,
# sentence-transformers and therefore the vector retrieval this chatbot now
# depends on. Same reason M3L2's install cell drops them.
!pip install --upgrade pip

# Core UI (stable with Python 3.12)
!pip install gradio==4.29.0
!pip install jinja2==3.1.4
!pip install fastapi==0.110.0
!pip install starlette==0.36.3

# LLM client + retrieval stack (must match M3L2 / M2L1)
%pip install openai==1.99.9
%pip install chromadb==1.5.9 sentence-transformers==2.7.0

### Import required libraries


In [1]:
import gradio as gr
import json
from typing import List, Tuple, Dict, Any, Iterator
from openai import OpenAI

# The real multi-agent workflow from Lesson 2, extracted into a module so both
# notebooks share one implementation. Importing it connects to the M2L1 Chroma
# index and loads the MiniLM query encoder, so this cell takes a few seconds.
from recommendation_workflow import run_workflow_from_profile

✓ Vector DB connected — 210 restaurants, 109 recipes, 125 known locations


In [2]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Define the model to use (must already be pulled: `ollama pull qwen2.5:7b`)
MODEL = "qwen2.5:7b"

## Build a basic chatbot

Start with a simple chatbot that echoes user messages. This will help you understand Gradio's chat components.

**Note**: You may see a warning notification indicating that a newer version of Gradio is available. You can safely ignore this warning and proceed with the lab.


In [3]:
from typing import List, Tuple
import gradio as gr

def echo_chatbot(message: str, history: List[Tuple[str, str]]) -> str:
    """A simple echo chatbot for testing."""
    return f"You said: {message}"

# Create interface
demo_echo = gr.ChatInterface(
    fn=echo_chatbot,
    title="Echo Chatbot",
    description="A simple chatbot that echoes your messages.",
    examples=[
        "Hello!",
        "How are you?",
        "Tell me about restaurants."
    ]
)

print("Basic chatbot interface created!")
print("To launch, run: demo_echo.launch()")

Basic chatbot interface created!
To launch, run: demo_echo.launch()


In [4]:
demo_echo.launch()

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


### **Question 1**: What two parameters does the chatbot function receive from Gradio's ChatInterface?


#### **Answer**: fn, title, description , examples


## Implement intent classification

Now, add logic to classify user intent: Do they want restaurant recommendations, recipe recommendations, or both?


In [5]:
def classify_intent(user_message: str, llm: Any = None) -> str:
    """Classify user intent as restaurant, recipe, both, or clarification.

    `llm` is unused — the module-level `client` above is what actually gets
    called. The parameter is kept so the existing test cells still run.
    """
    
    system_prompt = """You are an intent classifier for a food recommendation system.
    
Analyze the user's message and classify it as ONE of:
- "restaurant" - User wants restaurant recommendations
- "recipe" - User wants recipe recommendations
- "both" - User wants both restaurant and recipe recommendations
- "clarification" - User needs help or is asking a question
- "database" - User wants to add/edit/delete database entries

Examples:
"Where should I eat tonight?" → restaurant
"How do I make lasagna?" → recipe
"I want dinner ideas" → both
"What can you help me with?" → clarification
"I want to add a new restaurant" → database

Respond with ONLY the classification label."""
    
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    response = response.choices[0].message.content
    intent = response.strip().lower()
    
    # Validate intent
    valid_intents = ["restaurant", "recipe", "both", "clarification", "database"]
    if intent not in valid_intents:
        intent = "clarification"
    
    return intent

print("Intent classification function created!")

Intent classification function created!


In [6]:
# Test intent classification
test_messages = [
    "I'm looking for Italian restaurants",
    "How do I make pad thai?",
    "Give me dinner ideas",
    "What can you do?"
]

print("Testing intent classification:\n")
try:
    for msg in test_messages:
        intent = classify_intent(msg, client)
        print(f"Message: '{msg}'")
        print(f"Intent: {intent}\n")
except Exception as e:
    print(f"Intent classification requires valid OpenAI API key. Error: {e}")

Testing intent classification:

IMPORTANT: You are using gradio version 4.29.0, however version 4.44.1 is available, please upgrade.
--------
Message: 'I'm looking for Italian restaurants'
Intent: restaurant

Message: 'How do I make pad thai?'
Intent: recipe

Message: 'Give me dinner ideas'
Intent: both

Message: 'What can you do?'
Intent: clarification



## Extract user preferences

Once you know the user's intent, you need to extract their preferences from natural language.


In [7]:
def extract_preferences(user_message: str, llm: Any = None) -> Dict[str, Any]:
    """Extract user preferences from natural language input.

    `llm` is unused — see classify_intent above.
    """
    
    system_prompt = """You are a preference extractor for a food recommendation system.
    
Extract user preferences from their message and return JSON with these keys:
- favorite_cuisines: List of mentioned cuisines (e.g., ["Italian", "Thai"])
- dietary_restrictions: List of dietary needs (e.g., ["vegetarian", "gluten-free"])
- dining_occasion: Type of dining (e.g., ["casual", "fine dining", "quick bite"])
- price_range: Price preference (e.g., "$", "$$", "$$$", "$$$$")
- flavor_preferences: List of flavor preferences (e.g., ["spicy", "sweet"])
- preferred_location: The neighbourhood or city named in the message, copied
  verbatim. Use "" if no location is mentioned - do not guess.
- other_preferences: Any other relevant details (e.g. , ["gravy", "dry", "hot"])

If a field is not mentioned, use an empty list or "not specified".

Example:
Input: "I love spicy Thai food in Echo Park and I'm vegetarian"
Output: {
  "favorite_cuisines": ["Thai"],
  "dietary_restrictions": ["vegetarian"],
  "dining_occasion": "not specified",
  "price_range": "not specified",
  "flavor_preferences": ["spicy"],
  "preferred_location": "Echo Park",
  "other_preferences": "not specified"
}

Respond with ONLY valid JSON."""
    
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        response_format={"type": "json_object"}
    )
    response = response.choices[0].message.content
    
    try:
        preferences = json.loads(response)
    except:
        # Fallback if parsing fails
        preferences = {
            "favorite_cuisines": [],
            "dietary_restrictions": [],
            "dining_occasion": "not specified",
            "price_range": "not specified",
            "flavor_preferences": [],
            "preferred_location": "",
            "other_preferences": ""
        }
    
    return preferences

print("Preference extraction function created!")

Preference extraction function created!


### **Task 1**: Test the preference extraction function

Test the `extract_preferences` function with the message: "I'm looking for affordable vegetarian Mexican restaurants"


In [8]:
## Type your answer here

test_message = "I love spicy creamy Thai food and I'm vegetarian served hot and farm produced"
try:
    preferences = extract_preferences(test_message,client)
    print("Extracted Preferences:")
    print(json.dumps(preferences, indent=2))
except Exception as e:
    print(f"Requires valid OpenAI API key. Error: {e}")

Extracted Preferences:
{
  "favorite_cuisines": [
    "Thai"
  ],
  "dietary_restrictions": [
    "vegetarian"
  ],
  "dining_occasion": "not specified",
  "price_range": "not specified",
  "flavor_preferences": [
    "spicy",
    "creamy"
  ],
  "preferred_location": "not specified",
  "other_preferences": [
    "hot",
    "farm produced"
  ]
}


Take a screenshot of the Python code, clearly showing your implementation from Task 1. Name the screenshot ```M3L3_preference_extraction_test.jpg```.


## Integrate the multi-agent workflow

Now wire the chatbot to the **real** multi-agent workflow from Lesson 2 — the one that does an actual vector search over the M2L1 Chroma index and runs the six agents against `qwen2.5:7b`.

Two small adapters are needed between the chat layer and the workflow:

| Gap | Fix |
|---|---|
| `extract_preferences` returns `dining_occasion` (singular) and literal `"not specified"` strings; the workflow's `profile_to_query` expects `dining_occasions` and joins every value straight into the embedding query | `prefs_to_profile` renames the key and strips placeholder values so they don't pollute the search vector |
| Phase 4 returns only `{"name", "reasoning"}`, but the chat cards want cuisine, price, and location | `enrich_with_retrieved` joins each recommended name back to the record Phase 2 actually retrieved, recovering the real fields |

Because the chatbot already extracted a profile from the message, **Phase 1 is skipped** — `run_workflow_from_profile` starts at retrieval, saving one LLM call per turn.

In [9]:
PLACEHOLDERS = {"", "not specified", "none", "n/a", "unspecified"}


def _clean(value):
    """Drop placeholder values so they never reach the embedding query.

    profile_to_query() joins every profile value into one search string, so an
    unfiltered "not specified" would be embedded as if the user had asked for it.
    """
    if isinstance(value, list):
        cleaned = [str(v).strip() for v in value
                   if str(v).strip().lower() not in PLACEHOLDERS]
        return cleaned
    if value is None:
        return ""
    text = str(value).strip()
    return "" if text.lower() in PLACEHOLDERS else text


def prefs_to_profile(prefs: Dict[str, Any], user_message: str = "") -> Dict[str, Any]:
    """Adapt extract_preferences() output to the profile shape the workflow reads."""
    prefs = prefs or {}

    occasion = _clean(prefs.get("dining_occasion"))
    if isinstance(occasion, str):
        occasion = [occasion] if occasion else []

    # price_range and other_preferences are not read by profile_to_query, so
    # fold the useful part into summary to get it into the search query. This is
    # also how atmosphere words ("cozy", "soothing ambience") reach the search -
    # M2L1 embedded Vibe and Environment into each indexed document.
    other = _clean(prefs.get("other_preferences"))
    if isinstance(other, list):
        other = ", ".join(other)
    summary = " ".join(part for part in (user_message.strip(), other) if part)

    return {
        "favorite_cuisines": _clean(prefs.get("favorite_cuisines")) or [],
        "dietary_restrictions": _clean(prefs.get("dietary_restrictions")) or [],
        "dining_occasions": occasion,
        "flavor_preferences": _clean(prefs.get("flavor_preferences")) or [],
        "price_range": _clean(prefs.get("price_range")),
        "preferred_location": _clean(prefs.get("preferred_location")),
        "summary": summary,
    }


def enrich_with_retrieved(items: List[Dict[str, Any]],
                          retrieved: List[Dict[str, Any]],
                          fields: List[str]) -> List[Dict[str, Any]]:
    """Join Phase 4's {name, reasoning} back to the records Phase 2 retrieved.

    The synthesis agent returns names only; the real cuisine, price, location,
    atmosphere and similarity score are already sitting in state["retrieved_*"].
    Names that match nothing (an agent that drifted off the candidate list) are
    still returned, just without the extra fields.
    """
    by_name = {str(r.get("name", "")).strip().lower(): r for r in retrieved}

    enriched = []
    for item in items:
        name = str(item.get("name", "")).strip()
        record = by_name.get(name.lower(), {})
        merged = {"name": name, "reasoning": item.get("reasoning", "")}
        for field in fields:
            if record.get(field) not in (None, "", []):
                merged[field] = record[field]
        enriched.append(merged)
    return enriched


def run_recommendation_workflow(preferences: Dict[str, Any],
                                recommendation_type: str,
                                user_message: str = "") -> Dict[str, Any]:
    """Run the multi-agent workflow from Lesson 2 and return recommendations.

    Args:
        preferences: User preferences extracted from their message
        recommendation_type: "restaurant", "recipe", or "both"
        user_message: The raw message, used as the search fallback

    Returns:
        Dictionary with recommendations
    """
    print(f"Running workflow for {recommendation_type} recommendations...")

    profile = prefs_to_profile(preferences, user_message)

    # Phases 2-4: vector retrieval -> parallel analysis -> synthesis.
    # Phase 1 is skipped because extract_preferences already built the profile.
    state = run_workflow_from_profile(profile, user_message)

    recommendations = state.get("final_recommendations", {})
    if recommendations.get("error"):
        return {"error": recommendations["error"]}

    # Carry across every field the synthesis agent itself was shown, and let the
    # card decide which to display. signatures and ingredients are far too long
    # to print every time, but they are exactly what someone asking "what are
    # they known for?" or "what's in it?" wants to see.
    restaurants = enrich_with_retrieved(
        recommendations.get("restaurants", []),
        state.get("retrieved_restaurants", []),
        ["cuisine", "price_range", "location", "rating", "type",
         "signatures", "vibe", "environment", "similarity"],
    )
    recipes = enrich_with_retrieved(
        recommendations.get("recipes", []),
        state.get("retrieved_recipes", []),
        ["cuisine", "prep_time", "total_time", "servings",
         "ingredients", "description", "similarity"],
    )

    # Filter based on recommendation type
    if recommendation_type == "restaurant":
        return {"restaurants": restaurants}
    elif recommendation_type == "recipe":
        return {"recipes": recipes}
    else:  # both
        return {"restaurants": restaurants, "recipes": recipes}

print("Workflow integration function created!")

Workflow integration function created!


## Format the recommendations

Create a function to format recommendations in a user-friendly way.


In [10]:
import re


def format_price(value: Any) -> str:
    """structured_restaurant_data.json stores price_range as an int 1-4.
    Render it as the $-$$$$ scale the rest of the interface uses."""
    try:
        level = int(value)
    except (TypeError, ValueError):
        return "N/A"
    return "$" * level if 1 <= level <= 4 else "N/A"


def format_atmosphere(restaurant: Dict[str, Any]) -> str:
    """Combine the vibe and environment fields into one readable line.

    Both are part of the text M2L1 embedded, so they already influenced which
    restaurants were retrieved - this makes that visible in the answer.
    """
    vibe = str(restaurant.get("vibe", "") or "").strip()
    environment = str(restaurant.get("environment", "") or "").strip()
    parts = [p.rstrip(".") for p in (vibe, environment) if p]
    return " — ".join(parts) if parts else "N/A"


def format_list(values: Any, limit: int = 8) -> str:
    """Render a list field, capped so a long ingredient list can't swamp the card."""
    if not isinstance(values, list):
        text = str(values or "").strip()
        return text or "N/A"
    items = [str(v).strip() for v in values if str(v).strip()]
    if not items:
        return "N/A"
    shown = ", ".join(items[:limit])
    if len(items) > limit:
        shown += f", … (+{len(items) - limit} more)"
    return shown


# ---------------------------------------------------------------------------
# Adaptive card fields
# ---------------------------------------------------------------------------
# The card used to print the same rows no matter what was asked. These tables let
# it lead with the fields the question was actually about, and make it safe to
# carry long fields like ingredients and signatures - they only ever appear when
# someone asks for them.

# (field, which card it applies to, trigger words)
FIELD_TRIGGERS = [
    ("price_range", "restaurant", ("price", "cheap", "affordable", "budget",
                                   "expensive", "pricey", "cost")),
    ("location",    "restaurant", ("where", "near", "area",
                                   "neighbourhood", "neighborhood")),
    ("rating",      "restaurant", ("rating", "rated", "best", "top",
                                   "highly", "review")),
    ("atmosphere",  "restaurant", ("vibe", "ambience", "ambiance", "atmosphere",
                                   "cozy", "cosy", "romantic", "quiet", "lively",
                                   "date", "decor", "soothing", "relaxing")),
    ("signatures",  "restaurant", ("dish", "signature", "known for", "specialty",
                                   "speciality", "famous", "must try", "order")),
    ("type",        "restaurant", ("bistro", "cafe", "casual", "upscale",
                                   "fine dining", "kind of place")),
    ("cuisine",     "both",       ("cuisine", "food style", "kind of food")),
    ("total_time",  "recipe",     ("how long", "quick", "fast", "minutes",
                                   "hour", "time", "prep")),
    ("prep_time",   "recipe",     ("how long", "quick", "fast", "minutes",
                                   "hour", "time", "prep")),
    ("servings",    "recipe",     ("serve", "serving", "how many",
                                   "portion", "feed")),
    ("ingredients", "recipe",     ("ingredient", "what's in", "whats in",
                                   "contains", "made with", "allergen",
                                   "gluten", "dairy", "nut", "vegan",
                                   "vegetarian")),
    ("description", "recipe",     ("look like", "photo", "picture", "looks")),
]

# Rows shown on every card. Anything else is on-demand only.
DEFAULT_FIELDS = {
    "restaurant": ["cuisine", "price_range", "location", "rating", "atmosphere"],
    "recipe": ["cuisine", "total_time", "servings"],
}

FIELD_LABELS = {
    "restaurant": {
        # the restaurant records carry food_style (what the kitchen is known for),
        # not a clean cuisine label like the recipes do
        "cuisine": "Style",
        "price_range": "Price",
        "location": "Location",
        "rating": "Rating",
        "atmosphere": "Atmosphere",
        "signatures": "Known for",
        "type": "Type",
    },
    "recipe": {
        "cuisine": "Cuisine",
        "total_time": "Total time",
        "prep_time": "Prep time",
        "servings": "Servings",
        "ingredients": "Ingredients",
        "description": "Looks like",
    },
}


def _matches(text: str, triggers: tuple) -> bool:
    """Whole-word trigger match, tolerating a plural suffix.

    Plain substring matching is not safe here: "somewhere" contains "where",
    "minute" contains "nut", and "update" contains "date". The optional (e)s
    lets one entry cover dish/dishes and ingredient/ingredients.
    """
    return any(re.search(r"\b" + re.escape(t) + r"(?:e?s)?\b", text)
               for t in triggers)


def select_fields(user_message: str, preferences: Dict[str, Any] = None,
                  kind: str = "restaurant") -> List[str]:
    """Fields the user's own wording asked about, in trigger order.

    Deliberately a plain keyword match rather than another LLM call - a turn
    already costs four qwen2.5:7b calls, and this needs to be instant and
    predictable.
    """
    text = (user_message or "").lower()

    promoted = []
    for field, applies_to, triggers in FIELD_TRIGGERS:
        if applies_to not in (kind, "both"):
            continue
        if _matches(text, triggers):
            promoted.append(field)

    # Someone with a dietary restriction needs to see what is actually in the
    # dish, even when they never typed an ingredient word.
    if kind == "recipe":
        restrictions = (preferences or {}).get("dietary_restrictions") or []
        if restrictions and "ingredients" not in promoted:
            promoted.append("ingredients")

    return promoted


def card_fields(user_message: str, preferences: Dict[str, Any],
                kind: str) -> List[str]:
    """Requested fields first, then the standard rows that weren't already used."""
    promoted = select_fields(user_message, preferences, kind)
    tail = [f for f in DEFAULT_FIELDS[kind] if f not in promoted]
    return promoted + tail


def render_field(field: str, record: Dict[str, Any]) -> str:
    """Render one row's value, using the right formatter for its shape."""
    if field == "atmosphere":
        return format_atmosphere(record)
    if field == "price_range":
        return format_price(record.get(field))

    value = record.get(field)
    if isinstance(value, list):
        return format_list(value)
    text = str(value).strip() if value not in (None, "") else ""
    return text or "N/A"


def format_card(record: Dict[str, Any], index: int, kind: str,
                fields: List[str]) -> str:
    """One numbered recommendation, with its rows chosen by `fields`."""
    labels = FIELD_LABELS[kind]

    out = f"**{index}. {record.get('name', 'N/A')}**\n"
    for field in fields:
        text = render_field(field, record)
        # a standard row always prints (N/A included) so the card stays stable;
        # an on-demand row we simply don't have data for is skipped instead
        if text == "N/A" and field not in DEFAULT_FIELDS[kind]:
            continue
        out += f"   - {labels[field]}: {text}\n"
    out += f"   - Why: {record.get('reasoning', 'N/A')}\n\n"
    return out


def format_recommendations(recommendations: Dict[str, Any],
                           user_message: str = "",
                           preferences: Dict[str, Any] = None) -> str:
    """Format recommendations for display in the chat.

    `user_message` and `preferences` drive which rows appear; with neither, the
    cards fall back to the standard rows.
    """

    if recommendations.get("error"):
        return f"The recommendation workflow failed: {recommendations['error']}"

    output = ""

    # Format restaurant recommendations
    if "restaurants" in recommendations and recommendations["restaurants"]:
        fields = card_fields(user_message, preferences, "restaurant")
        output += "🍽️ **Restaurant Recommendations:**\n\n"
        for i, restaurant in enumerate(recommendations["restaurants"], 1):
            output += format_card(restaurant, i, "restaurant", fields)

    # Format recipe recommendations
    if "recipes" in recommendations and recommendations["recipes"]:
        fields = card_fields(user_message, preferences, "recipe")
        output += "👨‍🍳 **Recipe Recommendations:**\n\n"
        for i, recipe in enumerate(recommendations["recipes"], 1):
            output += format_card(recipe, i, "recipe", fields)

    if not output:
        output = "I couldn't generate recommendations. Please try again with more details about your preferences."

    return output

print("Formatting function created!")

Formatting function created!


## Build a complete chatbot

Now, combine everything into a complete chatbot.


In [11]:
def recommendation_chatbot(message: str, history: List[Tuple[str, str]]) -> Iterator[str]:
    """Main chatbot function that handles user requests.

    Written as a generator so Gradio can show progress: a recommendation turn
    now runs four qwen2.5:7b calls (classify -> extract -> 3 parallel analyses
    -> synthesis), which takes tens of seconds locally. ChatInterface accepts a
    generator and replaces the bubble with each successive yield.
    """

    try:
        # Step 1: Classify intent
        yield "🔎 Working out what you're after…"
        intent = classify_intent(message, client)
        print(f"Classified intent: {intent}")

        # Step 2: Handle different intents
        if intent == "clarification":
            yield """I'm your food recommendation assistant! I can help you with:

🍽️ **Restaurant recommendations** - Tell me your cuisine preferences, dietary restrictions, and occasion
👨‍🍳 **Recipe recommendations** - Let me know what you'd like to cook
📝 **Database management** - Add, update, or delete restaurants and recipes

Just describe what you're looking for, and I'll provide personalized recommendations!"""
            return

        elif intent == "database":
            yield """To manage the database, please use the tabs above:

- **Add Restaurant**: Submit a new restaurant
- **Add Recipe**: Submit a new recipe
- **Edit/Delete**: Modify or remove existing entries

Is there anything else I can help you with?"""
            return

        elif intent in ["restaurant", "recipe", "both"]:
            # Step 3: Extract preferences
            yield f"📝 Reading your preferences ({intent} request)…"
            preferences = extract_preferences(message, client)
            print(f"Extracted preferences: {preferences}")

            # Step 4: Run the real multi-agent workflow from Lesson 2
            yield "🧠 Searching the database and running the analysis agents… this takes a moment."
            recommendations = run_recommendation_workflow(preferences, intent, message)

            # Step 5: Format output. The message and preferences go along too, so
            # the cards can lead with whatever the question was actually about.
            yield format_recommendations(recommendations, message, preferences)
            return

        else:
            yield "I'm not sure how to help with that. Can you rephrase your request?"
            return

    except Exception as e:
        yield f"I encountered an error: {str(e)}. Please check that Ollama is running (`ollama serve`) and that `{MODEL}` has been pulled."


def chat_once(message: str) -> str:
    """Run one chat turn to completion and return the final reply.

    recommendation_chatbot yields progress updates for Gradio, so the test cells
    below drain the generator and keep the last value.
    """
    reply = ""
    for reply in recommendation_chatbot(message, []):
        pass
    return reply

print("Complete chatbot function created!")

Complete chatbot function created!


### **Question 2**: What are the five possible intent classifications the chatbot can identify?


#### **Answer**: You can use this cell to enter your answer.



## Add database management

Next, add functions for database management (add, update, and delete).


In [12]:
def add_restaurant(name: str, cuisine: str, price: str, location: str, description: str) -> str:
    """Add a new restaurant to the database."""
    # In a real implementation, this would add to the vector database
    print(f"Adding restaurant: {name}")
    return f"✅ Successfully added '{name}' to the database!"

def add_recipe(name: str, cuisine: str, difficulty: str, prep_time: str, ingredients: str, instructions: str) -> str:
    """Add a new recipe to the database."""
    # In a real implementation, this would add to the vector database
    print(f"Adding recipe: {name}")
    return f"✅ Successfully added '{name}' recipe to the database!"

print("Database management functions created!")

Database management functions created!


## Deploy and test

Next, create the complete Gradio interface with multiple tabs.


In [13]:
# Create the main interface with tabs
with gr.Blocks(title="Food Recommendation Chatbot", theme=gr.themes.Soft()) as demo:
    
    gr.Markdown("""
    # 🍽️ Food Recommendation Chatbot
    
    Your personal AI assistant for restaurant and recipe recommendations!
    """)
    
    with gr.Tabs():
        
        # Tab 1: Chat Interface
        with gr.Tab("💬 Chat"):
            chatbot_interface = gr.ChatInterface(
                fn=recommendation_chatbot,
                examples=[
                    "I'm looking for vegetarian restaurants",
                    "Suggest some easy recipes for dinner",
                    "I want spicy Thai food recommendations",
                    "What can you help me with?"
                ],
                title="Chat with the Recommendation Assistant",
                description=("Describe your food preferences and I'll recommend restaurants or recipes! "
                             "Recommendation requests run the full multi-agent workflow locally, "
                             "so give it a moment.")
            )
        
        # Tab 2: Add Restaurant
        with gr.Tab("➕ Add Restaurant"):
            gr.Markdown("### Add a New Restaurant to the Database")
            
            with gr.Row():
                with gr.Column():
                    rest_name = gr.Textbox(label="Restaurant Name")
                    rest_cuisine = gr.Textbox(label="Cuisine Type")
                    rest_price = gr.Dropdown(
                        choices=["$", "$$", "$$$", "$$$$"],
                        label="Price Range"
                    )
                with gr.Column():
                    rest_location = gr.Textbox(label="Location")
                    rest_description = gr.Textbox(
                        label="Description",
                        lines=3
                    )
            
            add_rest_btn = gr.Button("Add Restaurant", variant="primary")
            rest_output = gr.Textbox(label="Status")
            
            add_rest_btn.click(
                fn=add_restaurant,
                inputs=[rest_name, rest_cuisine, rest_price, rest_location, rest_description],
                outputs=rest_output
            )
        
        # Tab 3: Add Recipe
        with gr.Tab("➕ Add Recipe"):
            gr.Markdown("### Add a New Recipe to the Database")
            
            with gr.Row():
                with gr.Column():
                    recipe_name = gr.Textbox(label="Recipe Name")
                    recipe_cuisine = gr.Textbox(label="Cuisine Type")
                    recipe_difficulty = gr.Dropdown(
                        choices=["Easy", "Medium", "Hard"],
                        label="Difficulty"
                    )
                with gr.Column():
                    recipe_time = gr.Textbox(label="Prep Time")
                    recipe_ingredients = gr.Textbox(
                        label="Ingredients (comma-separated)",
                        lines=3
                    )
            
            recipe_instructions = gr.Textbox(
                label="Instructions",
                lines=5
            )
            
            add_recipe_btn = gr.Button("Add Recipe", variant="primary")
            recipe_output = gr.Textbox(label="Status")
            
            add_recipe_btn.click(
                fn=add_recipe,
                inputs=[recipe_name, recipe_cuisine, recipe_difficulty, recipe_time, recipe_ingredients, recipe_instructions],
                outputs=recipe_output
            )
        
        # Tab 4: About
        with gr.Tab("ℹ️ About"):
            gr.Markdown("""
            ## About This Chatbot
            
            This chatbot uses a multi-agent AI system to provide personalized food recommendations.
            
            ### Features:
            - 🤖 **Intelligent Agents**: Six specialized AI agents work together to analyze your preferences
            - 🔍 **Smart Search**: Vector database retrieval finds the most relevant options
            - 🎯 **Personalized**: Recommendations tailored to your tastes and dietary needs
            - 📝 **Editable Database**: Add your favorite restaurants and recipes
            
            ### How to Use:
            1. Go to the **Chat** tab
            2. Describe what you're looking for (cuisine, dietary restrictions, occasion, etc.)
            3. Receive personalized restaurant or recipe recommendations
            4. Use the **Add** tabs to contribute to the database
            
            ### Technologies:
            - **Ollama** running `qwen2.5:7b` locally, through the OpenAI-compatible client
            - **Chroma** + `all-MiniLM-L6-v2` embeddings for semantic search over the
              Module 2 index (210 restaurants, 109 recipes)
            - **ThreadPoolExecutor** to run the three Phase 3 analysis agents in parallel
            - **Gradio** for the user interface
            
            ### What happens on each recommendation:
            intent classification → preference extraction → vector retrieval →
            parallel trend / style / nutrition analysis → synthesis
            """)

print("Complete interface created!")
print("\nTo launch the chatbot, run: demo.launch()")

Complete interface created!

To launch the chatbot, run: demo.launch()


### Launch the chatbot


In [14]:
demo.launch(share=True)

Running on local URL:  http://127.0.0.1:7861
IMPORTANT: You are using gradio version 4.29.0, however version 4.44.1 is available, please upgrade.
--------
IMPORTANT: You are using gradio version 4.29.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://cbad3d89433162b4a0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


### **Question 3**: What are the four tabs in the complete Gradio interface?


#### **Answer**: You can use this cell to enter your answer.



## Test the chatbot

Test the chatbot logic with a sample interaction.


In [15]:
# Test the chatbot with a sample message
test_message = "I'm looking for spicy Indian non-vegetarian restaurants and recipes with a cozy vibe"

print("Testing chatbot with message:")
print(f"User: {test_message}\n")

try:
    response = chat_once(test_message)
    print("Bot Response:")
    print(response)
except Exception as e:
    print(f"Test requires Ollama running with {MODEL} pulled. Error: {e}")

Testing chatbot with message:
User: I'm looking for spicy Indian non-vegetarian restaurants and recipes with a cozy vibe

Classified intent: both
Extracted preferences: {'favorite_cuisines': ['Indian'], 'dietary_restrictions': [], 'dining_occasion': 'cozy', 'price_range': 'not specified', 'flavor_preferences': ['spicy'], 'preferred_location': 'not specified', 'other_preferences': 'not specified'}
Running workflow for both recommendations...

[Phase 2] Retrieving candidates from vector database...
  query: Indian, spicy, cozy, I'm looking for spicy Indian non-vegetarian restaurants and recipes with a cozy vibe
✓ Retrieved 10 restaurants and 20 recipes
    [0.604] The Neon Noodle — Thai (Van Nuys)
    [0.600] The Marigold Room — South Indian (Little India (Artesia))
    [0.575] The Copper Tandoor — truffle naan and lamb chops marinated in a secret yogurt-spice blend (Studio City)
    [0.489] Chicken Tikka Masala — Indian
    [0.474] Pad Thai — Thai
    [0.467] Thai Green Curry with Veget

### **Task 2**: Test the chatbot with a recipe request

Create a test message asking for recipe recommendations and run it through the chatbot.


In [16]:
## Type your answer here

test_recipe_message = "I want an easy gluten-free Mediterranean recipe I can make for dinner tonight"

print("Testing chatbot with recipe request:")
print(f"User: {test_recipe_message}\n")

try:
    response = chat_once(test_recipe_message)
    print("Bot Response:")
    print(response)
except Exception as e:
    print(f"Test requires Ollama running with {MODEL} pulled. Error: {e}")

Testing chatbot with recipe request:
User: I want an easy gluten-free Mediterranean recipe I can make for dinner tonight

Classified intent: recipe
Extracted preferences: {'favorite_cuisines': ['Mediterranean'], 'dietary_restrictions': ['gluten-free'], 'dining_occasion': 'not specified', 'price_range': 'not specified', 'flavor_preferences': [], 'preferred_location': 'not specified', 'other_preferences': []}
Running workflow for recipe recommendations...

[Phase 2] Retrieving candidates from vector database...
  query: Mediterranean, gluten-free, I want an easy gluten-free Mediterranean recipe I can make for dinner tonight
✓ Retrieved 10 restaurants and 20 recipes
    [0.393] The Velvet Olive — handmade chestnut agnolotti and osso buco (Los Gatos)
    [0.377] The Pasta Project — hand-rolled pasta, simple ingredients, daily-changing menu (Silver Lake)
    [0.363] The Green Leaf — plant-based (Venice Beach)
    [0.529] Greek Giouvertsa Salad with Lemon-Oregano Dressing — Greek
    [0.509]

### Congratulations!
You have successfully completed this lab.


### Summary

In this lab, you built a complete chatbot interface for your multi-agent recommendation system using Gradio. You:

1. Created a basic chat interface to understand Gradio components
2. Implemented intent classification to understand user requests
3. Built preference extraction to convert natural language to structured data
4. Integrated the multi-agent workflow from Lesson 2
5. Formatted recommendations for user-friendly display
6. Added database management features for adding restaurants and recipes
7. Created a multi-tab interface with chat, add forms, and information sections
8. Tested the chatbot with various user inputs

You now have a fully functional, production-ready chatbot that makes your AI system accessible to anyone, regardless of technical expertise. This completes your multi-agent recommendation system!


### Thank you for completing this lab!



## Authors 

[Tenzin Migmar](https://author.skills.network/instructors/tenzin_migmar)


©IBM Corporation. All rights reserved.


<!--## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2026-02-03|1|Tenzin Migmar|Create lab|
|2026-02-10|2|Jojy John|ID Reviewed|-->
